# Phase 1 — Part 3: Preprocessing

*Healthcare Stroke Prediction — Classification Project*

This notebook implements the preprocessing pipeline for the raw healthcare stroke dataset. It is the third deliverable of Phase 1 and produces the artefacts that Phase 2 (modelling) will consume.

## 3.1 Goals

The preprocessing stage must achieve the following, in order:

1. **Correct typing and basic cleaning** of the raw CSV, in particular recovering the numeric type of `bmi` (which is polluted by the string sentinel `"N/A"`), dropping the non-informative `id` column, and removing the single record where `gender == "Other"` (insufficient support for a dedicated category).
2. **Leakage-free transformation**. Every statistic used to transform the data — imputation medians, scaler means and standard deviations, one-hot category sets — must be fitted on the training split *only* and then applied to the test split. This is implemented with an `sklearn` `Pipeline` nested inside a `ColumnTransformer`, so that a single `fit` call learns everything and a single `transform` call reuses it.
3. **Stratified hold-out**. Because the positive class (`stroke == 1`) accounts for roughly 4.9% of the rows, the train/test split must be stratified on the target to preserve the base rate in both partitions.
4. **Reproducibility**. All random operations (split, SMOTE) are seeded with `random_state=42`. The fitted `ColumnTransformer` is persisted to disk so that any downstream notebook — including the modelling notebook in Phase 2 — can load and reapply the exact same transformation without re-deriving statistics.
5. **Optionality on class imbalance**. Resampling is a modelling decision, not a preprocessing decision. We therefore save the stratified split untouched, and additionally save a SMOTE-augmented training set as an independent artefact so that Phase 2 may choose between `class_weight`, SMOTE, or no resampling at all.

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False
    print('imblearn is not installed; the SMOTE section will be skipped.')

RANDOM_STATE = 42

In [2]:
DATA_PATH = Path('../data/healthcare-dataset-stroke-data.csv')

df = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df.shape}')

# Recover numeric bmi (the CSV encodes missing values as the string "N/A").
df['bmi'] = df['bmi'].replace('N/A', np.nan).astype(float)

# Drop the identifier column; it carries no predictive information.
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Drop the single row whose gender is encoded as 'Other'. A category of size
# one cannot be learned reliably and would inflate the one-hot dimension.
n_other = int((df['gender'] == 'Other').sum())
df = df.loc[df['gender'] != 'Other'].reset_index(drop=True)
print(f'Rows dropped for gender=="Other": {n_other}')
print(f'Cleaned shape: {df.shape}')
print(f'Missing bmi values: {int(df["bmi"].isna().sum())}')

Raw shape: (5110, 12)
Rows dropped for gender=="Other": 1
Cleaned shape: (5109, 11)
Missing bmi values: 201


## 3.2 Train / Test Split

We use an 80/20 split, stratified on the target. Stratification is essential here because the positive class is severely under-represented; a naive random split can drift the test-set prevalence by several percentage points, which would in turn make any downstream recall / PR-AUC estimate unreliable. A single deterministic split (`random_state=42`) is sufficient at this phase because cross-validation will be carried out on the training partition during modelling.

In [3]:
X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print('Train class distribution (proportion):')
print(y_train.value_counts(normalize=True).rename('proportion'))
print('Test class distribution (proportion):')
print(y_test.value_counts(normalize=True).rename('proportion'))

X_train: (4087, 10) | X_test: (1022, 10)
Train class distribution (proportion):
stroke
0    0.951309
1    0.048691
Name: proportion, dtype: float64
Test class distribution (proportion):
stroke
0    0.951076
1    0.048924
Name: proportion, dtype: float64


## 3.3 Feature Typing

The remaining eleven predictors split naturally into three continuous numeric variables and seven categorical variables. The two binary flags `hypertension` and `heart_disease` are already stored as integers, but we treat them as categorical so that they travel through the one-hot branch together with the other nominal fields; this keeps the scaler from applying a numerically meaningless standardisation to them.

**Numeric features** (`age`, `avg_glucose_level`, `bmi`): continuous, will be median-imputed and standardised.

**Categorical features** (`gender`, `hypertension`, `heart_disease`, `ever_married`, `work_type`, `Residence_type`, `smoking_status`): nominal, will be one-hot encoded.

In [4]:
numeric_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = [
    'gender',
    'hypertension',
    'heart_disease',
    'ever_married',
    'work_type',
    'Residence_type',
    'smoking_status',
]

assert set(numeric_features + categorical_features) == set(X_train.columns), (
    'Feature lists do not cover every column of X_train.'
)

## 3.4 Missing Value Treatment

Only `bmi` carries true missing values (approximately 3.9% of rows). We impute with the **training-set median**, for three reasons: (i) BMI is right-skewed, so the median is a more robust centre than the mean; (ii) the median is deterministic and does not introduce any inter-feature dependency, which keeps the preprocessing pipeline simple and auditable; (iii) preliminary EDA did not reveal a strong structural relationship between missingness and the target, so more aggressive schemes are unlikely to pay off.

Two alternatives are explicitly considered and left for ablation in Phase 2:

* **KNN imputation** (`sklearn.impute.KNNImputer`), which borrows BMI from observationally similar patients and can recover non-linear structure at the cost of being sensitive to the scaling of other features and to the choice of `k`.
* **Iterative imputation** (`sklearn.impute.IterativeImputer`), which regresses BMI on the other features in a round-robin fashion; more accurate in principle but also more expensive, less transparent, and potentially unstable on small classes.

`smoking_status` contains a substantial `"Unknown"` category (~30% of rows). We **retain** `"Unknown"` as a valid level rather than imputing it. Treating missingness as information is appropriate here because `"Unknown"` is concentrated in paediatric records and in records with incomplete intake forms; collapsing it into the modal category would wash out that signal. A defensive `SimpleImputer(most_frequent)` is nonetheless included in the categorical branch of the pipeline as a safeguard against genuinely unseen inputs at serving time.

## 3.5 Encoding

Categorical variables are one-hot encoded with `OneHotEncoder(handle_unknown='ignore', drop='if_binary')`. The `handle_unknown='ignore'` option guarantees that any category seen at test time but absent from the training split is encoded as an all-zero row instead of raising an exception; this is strictly necessary once the preprocessor is persisted and re-used in Phase 2. The `drop='if_binary'` option collapses each binary nominal field (e.g. `ever_married`, `Residence_type`) into a single indicator column, which reduces redundancy without discarding information. We intentionally avoid ordinal or target encoding: the categorical cardinalities are small (work_type has five levels, the rest are binary or ternary), so one-hot incurs no practical dimensionality cost, and target encoding would introduce a direct leakage risk.

## 3.6 Scaling

Numeric features are standardised with `StandardScaler`, i.e. shifted to zero mean and scaled to unit variance using statistics computed on the training split. Scaling is important for the two linear learners in the Phase 2 catalogue: (a) Logistic Regression with L2 regularisation penalises large coefficients, so features on different scales would be penalised unequally and the optimiser would converge more slowly; (b) Support Vector Machines rely on Euclidean distances in the RBF kernel and are extremely sensitive to the relative magnitudes of features — `avg_glucose_level` (up to ~270) would otherwise dominate `age` and `bmi` entirely. Tree-based learners are scale-invariant and will be unaffected by this step.

In [5]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='if_binary',
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# Fit on train only, then transform both partitions. This is where leakage
# prevention is operationalised.
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)

print(f'Fitted preprocessor output shape (train): {X_train_arr.shape}')
print(f'Fitted preprocessor output shape (test):  {X_test_arr.shape}')

Fitted preprocessor output shape (train): (4087, 17)
Fitted preprocessor output shape (test):  (1022, 17)


In [6]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_arr, columns=feature_names, index=X_train.index
)
X_test_processed = pd.DataFrame(
    X_test_arr, columns=feature_names, index=X_test.index
)

print(f'Number of output features: {len(feature_names)}')
print('First ten output feature names:')
for name in feature_names[:10]:
    print(f'  - {name}')
X_train_processed.head()

Number of output features: 17
First ten output feature names:
  - age
  - avg_glucose_level
  - bmi
  - gender_Male
  - hypertension_1
  - heart_disease_1
  - ever_married_Yes
  - work_type_Govt_job
  - work_type_Never_worked
  - work_type_Private


,age,avg_glucose_level,bmi,gender_Male,hypertension_1,heart_disease_1,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0.209397,-0.821221,0.549234,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3744,-0.629845,-0.485884,-0.988900,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4183,-0.364822,0.302317,-0.769166,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3409,-0.232310,0.062342,0.497532,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
284,-1.292405,-0.527297,0.355351,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


## 3.7 Class Imbalance Strategy

The target distribution is approximately 95% negative to 5% positive. There are three principled ways to cope with this during modelling:

1. **Cost-sensitive learning**, e.g. `class_weight='balanced'` in Logistic Regression or Random Forest. This re-weights the loss so that positives contribute as much as negatives, without fabricating data.
2. **Resampling at training time**, e.g. SMOTE, which synthesises new minority examples by interpolating between existing minority neighbours.
3. **Threshold tuning on the decision score**, which adjusts the operating point after fitting.

These choices belong to Phase 2, not to preprocessing, because they interact with the loss function and the cross-validation protocol. We therefore keep the stratified split untouched as the canonical artefact and, in addition, produce a SMOTE-augmented training set as an optional parallel artefact. **SMOTE is applied after the preprocessor and only to the training split**, never to the test split; applying SMOTE before the train/test split would leak synthesised information into the test data, and applying it to the test split would corrupt the evaluation.

In [7]:
if IMBLEARN_AVAILABLE:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(
        X_train_processed, y_train
    )
    X_train_smote = pd.DataFrame(
        X_train_smote, columns=X_train_processed.columns
    )
    y_train_smote = pd.Series(y_train_smote, name=y_train.name)

    print('Class counts before SMOTE:')
    print(y_train.value_counts())
    print('Class counts after SMOTE:')
    print(y_train_smote.value_counts())
else:
    X_train_smote, y_train_smote = None, None
    print('imblearn is not available; skipping SMOTE artefact.')

Class counts before SMOTE:
stroke
0    3888
1     199
Name: count, dtype: int64
Class counts after SMOTE:
stroke
0    3888
1    3888
Name: count, dtype: int64


## 3.8 Persistence

We persist the processed partitions as CSV files and the fitted `ColumnTransformer` as a joblib pickle. The rationale is twofold. First, separating preprocessing from modelling at an artefact boundary enforces that Phase 2 cannot accidentally peek at the raw data or re-fit statistics on the full dataset. Second, persisting the fitted transformer itself — not only its outputs — means that any future record (e.g. a held-out validation set, a production request) can be transformed through the exact same learned mapping by a single `joblib.load(...).transform(...)` call.

In [8]:
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

written = []

X_train_processed.to_csv(OUTPUT_DIR / 'X_train.csv', index=False)
written.append('X_train.csv')

X_test_processed.to_csv(OUTPUT_DIR / 'X_test.csv', index=False)
written.append('X_test.csv')

y_train.to_frame().to_csv(OUTPUT_DIR / 'y_train.csv', index=False)
written.append('y_train.csv')

y_test.to_frame().to_csv(OUTPUT_DIR / 'y_test.csv', index=False)
written.append('y_test.csv')

if IMBLEARN_AVAILABLE and X_train_smote is not None:
    X_train_smote.to_csv(OUTPUT_DIR / 'X_train_smote.csv', index=False)
    written.append('X_train_smote.csv')
    y_train_smote.to_frame().to_csv(
        OUTPUT_DIR / 'y_train_smote.csv', index=False
    )
    written.append('y_train_smote.csv')

joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib')
written.append('preprocessor.joblib')

print('Artefacts written to', OUTPUT_DIR.resolve())
for name in written:
    print(f'  - {name}')

Artefacts written to C:\Users\donat\Desktop\Markie\outputs
  - X_train.csv
  - X_test.csv
  - y_train.csv
  - y_test.csv
  - X_train_smote.csv
  - y_train_smote.csv
  - preprocessor.joblib


## 3.9 Reproducibility Summary

| Item | Value |
|---|---|
| Random state | `42` (train/test split and SMOTE) |
| Split protocol | Stratified 80 / 20 on `stroke` |
| Rows dropped (`gender == "Other"`) | logged in cell 3.1 above |
| Missing `bmi` handling | Median imputation fitted on train only |
| `smoking_status` `"Unknown"` handling | Retained as its own level |
| Scaler | `StandardScaler`, fitted on train only |
| Encoder | `OneHotEncoder(handle_unknown='ignore', drop='if_binary')`, fitted on train only |
| Resampling | Not applied to canonical split; optional SMOTE artefact provided |
| Persisted preprocessor | `../outputs/preprocessor.joblib` |
| Persisted splits | `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` |
| Optional SMOTE split | `X_train_smote.csv`, `y_train_smote.csv` (when `imblearn` is installed) |

Any downstream notebook can reconstruct the exact transformation used here by loading `preprocessor.joblib` and calling `transform` on a new frame with the same column layout. The raw cleaning steps (BMI cast, `id` drop, `"Other"` gender drop) are encapsulated in `src/preprocessing.py::load_and_clean` for programmatic re-use.